In [1]:
from datasets import load_dataset

In [2]:
imdb_dataset = load_dataset("imdb")

split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)

imdb_train_set, imdb_valid_set = split['train'], split['test']

imdb_test_set = imdb_dataset['test']

In [3]:
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
imdb_train_set

Dataset({
    features: ['text', 'label'],
    num_rows: 20000
})

In [5]:
imdb_train_set[1]

{'text': "'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!",
 'label': 1}

**Tokenization using the hugging face tokenizers library:**

In [6]:
#here's how to train a BPE (Byte- Pair encoding) model on the IMDB movies reviews dataset
import tokenizers

In [7]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>","<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens=special_tokens)
train_reviews = [review["text"].lower() for review in imdb_train_set]

In [8]:
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

Now, using our BPE tokenizer to tokenize some text

In [9]:
some_review = "what an awesome movie!"

bpe_encoding = bpe_tokenizer.encode(some_review)

In [10]:
bpe_encoding

Encoding(num_tokens=7, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [11]:
bpe_encoding.tokens

['what', 'an', 'aw', 'es', 'ome', 'movie', '!']

In [12]:
bpe_encoding.ids

[303, 139, 373, 149, 240, 211, 4]

In [13]:
bpe_tokenizer.decode(bpe_encoding.ids)

'what an aw es ome movie !'

the tokenizer keeps track of each token's start and end offset in the original string

In [14]:
bpe_encoding.offsets

[(0, 4), (5, 7), (8, 10), (10, 12), (12, 15), (16, 21), (21, 22)]

In [15]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=281, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=114, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=285, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [16]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [17]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])

In [19]:
import torch

In [21]:
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])

In [23]:
bpe_batch_ids.shape

torch.Size([3, 285])

In [24]:
bpe_batch_ids

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

each encoding object also includes an attention_mask attribute containing a 1 for each nonpadding token and a 0 for each padding token. this can be used inside the model to easily ignore the padded timesteps. just by multiplying a tensor with the attention mask. 

In [26]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])

In [27]:
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [28]:
lengths = attention_mask.sum(dim=-1)
lengths

tensor([281, 114, 285])

byte-level pretokenizer: it replaces all spaces with a special character 

simply reusing a pretrained tokenizer.

In [29]:
# using the BBPE tokenizer used in the GPT2 model
import transformers

In [30]:
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True, max_length=500)

In [31]:
gpt2_encoding

{'input_ids': [[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 10508, 355, 366, 3847, 2802, 11074, 9785, 1681, 46390, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 991, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 262, 16585, 1022, 285, 40302, 269, 5718, 290, 33826, 8803, 302, 44655, 318, 2407, 10457, 13, 262, 17262, 286, 511, 2776, 389, 6452, 13, 269, 5718, 318, 9623, 355, 1464, 11, 290, 302, 44655, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 72, 22127, 326, 1312, 1053, 1239, 1775, 4173, 64, 443, 7114, 338, 711, 11, 475, 1312, 3285, 326, 474, 323, 1803, 261, 477, 268, 338, 16711, 318, 17074, 13, 262, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 22595, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159, 29847, 1

In [33]:
gpt2_token_ids = gpt2_encoding['input_ids'][0][:10]
gpt2_token_ids

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]

In [34]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

if you prefer to use a pretrained wordpiece tokenizer, you can reuse the tokenizer of any LLM that was pretrained using wordpiece such as BERT. due to the nature of the initial task used to train the models, the bert tokenizer has a padding token while the gpt2 one does not. so we can specify padding=True when encofing a batch of reviews: and we can also specvify return tensors=pt to get a pytorch tensor instead of a list of python list of losts

In [36]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True, truncation=True, max_length=500, return_tensors="pt")

the resulting token ids and attention masks are nicely padded tensors

In [38]:
bert_encoding["input_ids"].shape

torch.Size([3, 234])

In [39]:
bert_encoding["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [40]:
bert_encoding["input_ids"]

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

bert model - by default the tokenizer starts with token 101 ([CLS]) and ends with token 102 ([SEP]). These tokens are needed by the BERT model but unless your model needs them too, you can drop them by setting add_special_tokens to False when calling the tokenizer.

In [41]:
bert_tokenizer2 = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding2 = bert_tokenizer2(train_reviews[:3], padding=True, add_special_tokens=False, truncation=True, max_length=500, return_tensors="pt")

In [42]:
bert_encoding2["input_ids"]

tensor([[ 2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,  2411,
          2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,  2872,
          2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,  1000,
          1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,  1996,
          2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,  3861,
          3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,  1005,
          1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,  1012,
          1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,  3243,
          8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,  1012,
         19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,  2028,
          1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,  7987,
          1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,  1045,
          1005,  2310,  2196,  2464, 11209, 20206,  

the transformers library also provides an object that can wrap your own tokenizer. (from the tokenizers library) and give it the same API as the pretrained tokenizers. 

In [43]:
hf_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object=bpe_tokenizer)

**Building and Training the sentiment analysis model:**

In [44]:
import transformers
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
bert_model.embeddings.word_embeddings

Embedding(30522, 768, padding_idx=0)

In [45]:
bert_model.embeddings

BertEmbeddings(
  (word_embeddings): Embedding(30522, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (token_type_embeddings): Embedding(2, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [47]:
import torch.nn as nn

In [48]:
weights = bert_model.embeddings.word_embeddings.weight.data

new_emb_layer = nn.Embedding.from_pretrained(weights, freeze=True)

In [49]:
train_reviews[:3]

['stage adaptations often have a major fault. they often come out looking like a film camera was simply placed on the stage (such as "night mother"). sidney lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. still, it\'s nice to look at for what it is. the chemistry between michael caine and christopher reeve is quite brilliant. the dynamics of their relationship are surprising. caine is fantastic as always, and reeve gets one of his few chances to really act.<br /><br />i confess that i\'ve never seen ira levin\'s play, but i hear that jay presson allen\'s adaptation is faithful. the script is incredibly convoluted, and keeps you guessing. "deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10',
 "'the rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them r

In [50]:
# for example, why not reuse the entire pretrained BERT model for our sentiment analysis model. to use the bert model, the transformers libary lets us call it like a function, passing it the tokenized reviews
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True, max_length=200, truncation=True, return_tensors="pt")

In [53]:
dict(bert_encoding).keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

In [54]:
bert_output = bert_model(**bert_encoding)

In [55]:
bert_output

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.2470, -0.4977,  0.1271,  ...,  0.0853,  0.5811,  0.4216],
         [ 0.0992,  0.4679, -0.2322,  ..., -0.0807,  0.8464, -0.6777],
         [ 0.9692,  0.2901, -0.1934,  ..., -0.1843,  0.0797,  0.0119],
         ...,
         [ 0.0884,  0.3943, -0.0035,  ...,  0.4302,  0.3150, -0.1580],
         [-0.0762, -0.2808,  0.8119,  ...,  0.4624,  0.8381, -0.3956],
         [ 0.2624, -0.2101,  0.6697,  ...,  0.4399,  0.6036, -0.3578]],

        [[-0.0101, -0.3241,  0.3658,  ..., -0.4497,  0.5121, -0.1260],
         [-0.2964, -0.2872, -0.5425,  ...,  0.6877,  1.3121, -1.5069],
         [-0.4449, -0.8080, -0.4515,  ...,  0.5536,  0.8961, -0.3652],
         ...,
         [ 0.3836, -0.1907,  0.3335,  ..., -0.0445, -0.1109,  0.0854],
         [ 0.3802,  0.0156,  0.4263,  ...,  0.1164,  0.1199, -0.2163],
         [ 0.1742, -0.0786,  0.5463,  ..., -0.0133, -0.2344, -0.1866]],

        [[ 0.0737, -0.7819,  0.1337,  ...,  0.2590,  

In [56]:
bert_output.last_hidden_state.shape

torch.Size([3, 200, 768])

bert output includes the last_hidden_state which contains the contextualised embeddings for each token. the word 'last' in this state referes to the last layer. when using bert, one other option is to only use the contextualised embedding for the very first oken, which is the class token [CLS]. Indeed, during pretraining, the bert model had to perform a text classification task based solely on this token's contextualised embedding. As a result, it learned to summarise the most important features of the text into this embedding. 

In [58]:
# so using:
bert_output.last_hidden_state[:, 0].shape

torch.Size([3, 768])

In fact, the bert model contains an extra hidden layer on top of the class embedding composed of an nn.Linear model and an nn.Tanh module. this hidden layer is called the pooler. to use it replace last_hidden_state with pooler_output. 

**Task-Specific Classes:**
- to tackle our binary classification task using BERT, we can use the BertForSequenceClassification class provided by the transformers library. it is just a bert model plus a classification head on top of it. 

In [59]:
from transformers import BertForSequenceClassification

In [62]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [63]:
torch.manual_seed(42)

bert_for_binary_clf = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2, dtype=torch.float16).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [64]:
bert_for_binary_clf

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [67]:
for a, b in bert_for_binary_clf.named_children():
    print(a)

bert
dropout
classifier


transformers library contains many task specific classes based on various pretrained models such as BertForQuestionAnswering, or RobertaForSequenceClassication. can also use AutoModelForSequenceClassification to let the library pick the right class. 

In [68]:
encoding = bert_tokenizer(["This was a great movie!"])

with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids = torch.tensor(encoding["input_ids"], device=device),
        attention_mask = torch.tensor(encoding["attention_mask"], device=device)
    )

In [69]:
output.logits

tensor([[-0.0122,  0.6313]], device='mps:0', dtype=torch.float16)

In [70]:
torch.softmax(output.logits, dim=-1)

tensor([[0.3445, 0.6558]], device='mps:0', dtype=torch.float16)

In [71]:
output.logits.shape

torch.Size([1, 2])

using bert for sequence classification this way or automodel for sequence classification this way, the head of the model isn't trained and should be finetuned on the data. 

**HF Pipelines**
- the transformers lib provides a very convenient api to download and use pretrained pipelines for various tasks. each pipeline contains a pretrained model along with its corresponding preprocessing and post processing modules

In [72]:
# e.g. a sentiment analysis pipeline
from transformers import pipeline

In [74]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model = model_name, truncation=True, max_length=512)

Device set to use mps:0


In [76]:
classifier_imdb(train_reviews[:10])

[{'label': 'POSITIVE', 'score': 0.9996108412742615},
 {'label': 'POSITIVE', 'score': 0.9998623132705688},
 {'label': 'NEGATIVE', 'score': 0.9943684935569763},
 {'label': 'POSITIVE', 'score': 0.997913658618927},
 {'label': 'POSITIVE', 'score': 0.999544084072113},
 {'label': 'NEGATIVE', 'score': 0.9845332503318787},
 {'label': 'POSITIVE', 'score': 0.9859278202056885},
 {'label': 'POSITIVE', 'score': 0.9993758797645569},
 {'label': 'POSITIVE', 'score': 0.9978922009468079},
 {'label': 'NEGATIVE', 'score': 0.9997020363807678}]

In [77]:
model_name = "huggingface/distilbert-base-uncased-finetuned-mnli"

classifier_mnli = pipeline("text-classification", model=model_name)

Device set to use mps:0


In [78]:
classifier_mnli([
    "She loves me. [SEP] She loves me not. [SEP]",
    "Alice just woke up. [SEP] Alice is awake. [SEP]",
    "I like dogs. [SEP] Everyone likes dogs. [SEP]"
])

[{'label': 'contradiction', 'score': 0.9717152714729309},
 {'label': 'entailment', 'score': 0.9119167327880859},
 {'label': 'neutral', 'score': 0.9509280323982239}]